# Exploratory Data Analysis: Telco Customer Churn Dataset

## Executive Summary

This notebook provides a comprehensive exploratory data analysis (EDA) of the telecommunications customer churn dataset. The analysis follows data science methodologies including statistical profiling, feature engineering assessment, and business impact quantification.

## Objectives

1. **Data Quality Assessment**: Comprehensive validation of data integrity, completeness, and consistency
2. **Statistical Profiling**: Distribution analysis, outlier detection, and correlation assessment
3. **Business Intelligence**: Customer behavior patterns and churn factor identification
4. **Feature Engineering Insights**: Variable transformation recommendations for downstream modeling
5. **Hypothesis Generation**: Statistical hypotheses for further testing and validation

## Dataset Specifications

- **Source**: Telco Customer Churn Dataset (IBM Watson Analytics)
- **Observations**: 7,043 customer records
- **Features**: 21 attributes spanning demographics, service subscriptions, and financial metrics
- **Target Variable**: Binary churn indicator (Yes/No)
- **Data Collection Period**: Cross-sectional snapshot
- **Business Domain**: Telecommunications service provider

## Methodology

This analysis implements the CRISP-DM methodology with emphasis on:
- Automated data profiling using pandas-profiling and ydata-profiling
- Statistical hypothesis testing for feature significance
- Visualization techniques using plotly and seaborn
- Business impact quantification through cohort analysis


In [ ]:
# Environment Setup and Configuration
import warnings
warnings.filterwarnings('ignore')

# Standard Library
import sys
import os
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass

# Data Manipulation and Analysis
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, cramers_v
import missingno as msno

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistical Analysis
import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Project Utilities
sys.path.append(str(Path.cwd().parent))
from utils.data_preprocessing import load_telco_data, prepare_data_for_analysis

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# Plotting Configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9

print("Environment successfully configured")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")


## Data Loading and Initial Assessment

### Data Ingestion Protocol

Implementation of robust data loading with comprehensive validation checks and initial quality assessment metrics.


In [ ]:
# Data Loading and Quality Assessment
@dataclass
class DataQualityMetrics:
    """Data quality assessment metrics"""
    total_records: int
    total_features: int
    missing_values: Dict[str, int]
    duplicate_records: int
    data_types: Dict[str, str]
    memory_usage: float

def load_and_assess_data() -> Tuple[pd.DataFrame, DataQualityMetrics]:
    """Load data and perform comprehensive quality assessment"""
    
    # Load data using utility function
    df = load_telco_data('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
    
    # Calculate quality metrics
    quality_metrics = DataQualityMetrics(
        total_records=len(df),
        total_features=len(df.columns),
        missing_values=df.isnull().sum().to_dict(),
        duplicate_records=df.duplicated().sum(),
        data_types=df.dtypes.astype(str).to_dict(),
        memory_usage=df.memory_usage(deep=True).sum() / 1024**2  # MB
    )
    
    return df, quality_metrics

# Execute data loading
df, quality_metrics = load_and_assess_data()

# Display quality assessment
print("="*80)
print("DATA QUALITY ASSESSMENT REPORT")
print("="*80)
print(f"Dataset Dimensions: {quality_metrics.total_records:,} records × {quality_metrics.total_features} features")
print(f"Memory Utilization: {quality_metrics.memory_usage:.2f} MB")
print(f"Duplicate Records: {quality_metrics.duplicate_records}")

print("\nMissing Value Analysis:")
missing_summary = pd.Series(quality_metrics.missing_values)
print(missing_summary[missing_summary > 0])

print("\nData Type Distribution:")
dtype_counts = pd.Series(quality_metrics.data_types).value_counts()
print(dtype_counts)

# Display first few records
print("\nDataset Preview:")
print(df.head().to_string())
